# 02.2 — TF-IDF

**Problem it solves:** Boolean retrieval ranks nothing — all matching docs are equal. TF-IDF ranks documents by *how relevant* they are to the query.

**Core insight:** A term that appears often in a document (high TF) but rarely across all documents (high IDF) is a strong signal of relevance. 'cancer' in a medical paper is more informative than 'the'.

**TF(t,d)** = frequency of term t in document d
**IDF(t)** = log(N / df(t)) — N docs, df(t) docs contain term t
**TF-IDF(t,d)** = TF(t,d) × IDF(t)

---

In [ ]:
import math
import numpy as np
from collections import Counter
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r'\b[a-z]+\b', text.lower())

class TFIDFVectorizer:
    """
    TF-IDF vectorizer from scratch.
    Produces sparse document vectors for cosine similarity search.
    """
    
    def __init__(self, min_df: int = 1, max_df_ratio: float = 1.0,
                 sublinear_tf: bool = False):
        """
        sublinear_tf: use 1 + log(tf) instead of raw tf.
        Prevents a document with 'cat' x100 from dominating.
        """
        self.min_df = min_df
        self.max_df_ratio = max_df_ratio
        self.sublinear_tf = sublinear_tf
        self.vocab: dict[str, int] = {}
        self.idf: np.ndarray = None
        self.n_docs: int = 0
    
    def fit(self, docs: list[str]):
        self.n_docs = len(docs)
        
        # Compute document frequencies
        df = Counter()
        for doc in docs:
            terms = set(tokenize(doc))
            df.update(terms)
        
        # Filter by min_df and max_df_ratio
        max_df = int(self.max_df_ratio * self.n_docs)
        valid_terms = [
            term for term, freq in sorted(df.items())
            if self.min_df <= freq <= max_df
        ]
        
        self.vocab = {term: i for i, term in enumerate(valid_terms)}
        
        # Compute IDF: log((N+1)/(df+1)) + 1  [sklearn's smoothed IDF]
        self.idf = np.array([
            math.log((self.n_docs + 1) / (df[term] + 1)) + 1
            for term in valid_terms
        ])
        
        return self
    
    def transform(self, docs: list[str]) -> np.ndarray:
        """Convert docs to TF-IDF matrix (n_docs x vocab_size)."""
        matrix = np.zeros((len(docs), len(self.vocab)))
        
        for i, doc in enumerate(docs):
            tokens = tokenize(doc)
            tf_counts = Counter(tokens)
            doc_len = len(tokens)
            
            for term, count in tf_counts.items():
                if term not in self.vocab:
                    continue
                j = self.vocab[term]
                
                # Raw TF or sublinear
                tf = (1 + math.log(count)) if self.sublinear_tf else count / doc_len
                matrix[i, j] = tf * self.idf[j]
        
        # L2 normalize (unit vectors for cosine similarity)
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1  # avoid div by zero
        return matrix / norms
    
    def fit_transform(self, docs: list[str]) -> np.ndarray:
        return self.fit(docs).transform(docs)

# Corpus
docs = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks with many layers",
    "natural language processing analyzes human text and speech",
    "machine translation converts text from one language to another",
    "neural networks are used in image recognition and NLP",
    "the quick brown fox jumps over the lazy dog",
    "artificial intelligence research has grown rapidly",
]

vec = TFIDFVectorizer(min_df=1, sublinear_tf=True)
tfidf_matrix = vec.fit_transform(docs)

print(f"Vocab size: {len(vec.vocab)}")
print(f"Matrix shape: {tfidf_matrix.shape}")

# Show top TF-IDF terms for doc 0
doc_vec = tfidf_matrix[0]
top_terms = sorted(vec.vocab.items(), key=lambda x: doc_vec[x[1]], reverse=True)[:8]
print(f"\nTop terms for doc 0: '{docs[0]}'")
for term, idx in top_terms:
    print(f"  {term:20} tfidf={doc_vec[idx]:.4f}  idf={vec.idf[idx]:.4f}")

In [ ]:
# TF-IDF Search: cosine similarity between query vector and doc vectors

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    # Both already L2-normalized in our vectorizer, so dot product = cosine
    return float(np.dot(a, b))

def search(query: str, vectorizer: TFIDFVectorizer, 
           doc_matrix: np.ndarray, docs: list[str], 
           top_k: int = 5) -> list[tuple]:
    query_vec = vectorizer.transform([query])[0]
    scores = [(cosine_similarity(query_vec, doc_matrix[i]), i) 
              for i in range(len(docs))]
    scores.sort(reverse=True)
    return [(score, doc_id, docs[doc_id]) for score, doc_id in scores[:top_k]]

queries = [
    "neural network deep learning",
    "language translation",
    "artificial intelligence",
    "fox dog animal",  # no good match
]

for q in queries:
    print(f"\nQuery: {q!r}")
    results = search(q, vec, tfidf_matrix, docs)
    for score, doc_id, text in results[:3]:
        print(f"  [{doc_id}] score={score:.4f}  {text}")

In [ ]:
# Visualize IDF weights: which terms are 'informative'?
# High IDF = rare = informative
# Low IDF = common = not very informative (like stopwords)

idf_by_term = sorted(zip(vec.vocab.keys(), vec.idf), key=lambda x: x[1])

print("Lowest IDF (most common, least informative):")
for term, idf_val in idf_by_term[:10]:
    print(f"  {term:20} IDF={idf_val:.3f}")

print("\nHighest IDF (rarest, most informative):")
for term, idf_val in idf_by_term[-10:]:
    print(f"  {term:20} IDF={idf_val:.3f}")

In [ ]:
# TF-IDF failure: long documents get penalized
# A 1000-word document with 'neural' appearing 5 times has lower TF
# than a 50-word document with 'neural' appearing 2 times.
# (5/1000=0.005 vs 2/50=0.04)
# This is the core problem BM25 fixes.

doc_short = "neural network is great"
doc_long  = "neural network is great " + "and " * 100 + "works well"

test_vec = TFIDFVectorizer().fit([doc_short, doc_long])
query_vec = test_vec.transform(["neural network"])[0]
short_vec = test_vec.transform([doc_short])[0]
long_vec  = test_vec.transform([doc_long])[0]

print("Document length sensitivity:")
print(f"  Short doc ({len(doc_short.split())} words): score={cosine_similarity(query_vec, short_vec):.4f}")
print(f"  Long doc  ({len(doc_long.split())} words): score={cosine_similarity(query_vec, long_vec):.4f}")
print("\nShort doc wins despite both containing 'neural network' once.")
print("BM25 uses document length normalization to fix this.")

## Summary

**What it does:** Converts documents into vectors where dimensions are terms and values represent term importance (frequent in doc, rare across corpus).

**When to use:** Baseline retrieval, document similarity, feature extraction for classical ML classifiers.

**What breaks it:**
- Document length bias — longer docs score lower for the same content
- Term frequency saturation — TF grows linearly even when one more occurrence adds diminishing information
- No semantic understanding — 'car' and 'automobile' are unrelated in TF-IDF space
- Exact match only — no morphological variants unless you stem first

BM25 fixes the first two problems. Word embeddings fix the last two.

---
**Next:** 02.3 — BM25: the gold standard for classical retrieval